# 02 - BraTS 2020 test cases -> MNI space (same as the HCP training scans)

**Press Run All.** Needs: Accelerator = None, Internet = On, a BraTS 2020 dataset attached as Input (the public Kaggle one with the `BraTS20_Training_xxx` folders), and the Kaggle Secret `GITHUB_TOKEN`.

What it does: installs FSL -> downloads our code -> for the 31 frozen test cases: reorient, resample to 1 mm, register the T2 to MNI152, apply the same transform to the tumor outline (nearest neighbour), scale brightness exactly like the HCP scans -> picture grid + summary table -> zips results.
Check the `volume_ratio` column: the tumor volume before/after registration should be about 1.0.

## 0. Settings

In [ ]:
LIMIT = 2           # number of test cases to process: 2 = quick test, None = all 31
WORKERS = None      # None = automatic (CPU cores - 1, at most 3)

## 1. Install FSL (FLIRT + MNI templates) - a few minutes

In [ ]:
import os, subprocess, sys
def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd[:80]}")
    return r.returncode

sh("mkdir -p /tmp/mm && cd /tmp/mm && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba", check=True)
sh("/tmp/mm/bin/micromamba create -y -p /tmp/fsl-env -c https://fsl.fmrib.ox.ac.uk/fsldownloads/fslconda/public/ -c conda-forge fsl-flirt fsl-data_standard", check=True)

os.environ["FSLDIR"] = "/tmp/fsl-env"
os.environ["FSLOUTPUTTYPE"] = "NIFTI_GZ"
os.environ["PATH"] = "/tmp/fsl-env/bin:" + os.environ["PATH"]

In [ ]:
import shutil
assert shutil.which("flirt"), "FSL tools not found: the install above failed"
MNI = "/tmp/fsl-env/data/standard/MNI152_T1_1mm_brain.nii.gz"
assert os.path.exists(MNI), f"MNI template missing: {MNI}"
print("FSL OK:", shutil.which("flirt"), "| template OK")

## 2. Get our code from the private GitHub repo (needs the `GITHUB_TOKEN` secret)

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "/kaggle/working/repo"
sh(f"cd /kaggle/working && rm -rf repo && git clone -q https://{token}@github.com/PiyushGarg0146/lf-synthseg-btp.git repo", check=True)
sh(f"{sys.executable} -m pip install -q nibabel scipy matplotlib pyyaml scikit-learn")
sh(f"cd {REPO} && git log --oneline | head -3 && ls configs", check=True)

## 3. Check that the BraTS data is attached and contains the 31 test cases

In [ ]:
sys.path.insert(0, REPO + "/scripts"); sys.path.insert(0, REPO)
import fsl_brats as B
ids = B.read_case_ids(REPO + "/configs/test_split_brats2020.txt")
print(len(ids), "test cases in the frozen split, e.g.", ids[:3])
found = B.find_case_files("/kaggle/input", ids)       # raises with a clear message if any case is missing
print("found all", len(found), "cases. Example:", [p.name for p in found[ids[0]]])

## 4. Register the test cases to MNI space

In [ ]:
SRC = "/kaggle/input"
OUT = "/kaggle/working/brats_processed"
sh(f"rm -rf {OUT}")
cores = os.cpu_count() or 1
workers = WORKERS or max(1, min(3, cores - 1))
print(f"CPU cores: {cores}, workers: {workers}")
cmd = (f"cd {REPO} && {sys.executable} -u scripts/fsl_brats.py --src {SRC} --cases configs/test_split_brats2020.txt "
       f"--dst {OUT} --workers {workers}" + (f" --limit {LIMIT}" if LIMIT else ""))
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()
assert p.returncode == 0, "pipeline failed - see the messages above"

## 5. Check the results

In [ ]:
import glob, numpy as np, nibabel as nib
print(open(OUT + "/summary.csv").read())
for f in sorted(glob.glob(OUT + "/*/t2.nii.gz")):
    d = f.rsplit("/", 1)[0]
    t = nib.load(f); tm = np.asarray(nib.load(d + "/tumor.nii.gz").dataobj) > 0
    print(d.rsplit("/", 1)[1], "shape", t.shape, "voxel", t.header.get_zooms()[:3], "orientation", nib.aff2axcodes(t.affine),
          "| tumor touches image border:", bool(tm[0].any() or tm[-1].any() or tm[:, 0].any() or tm[:, -1].any() or tm[:, :, 0].any() or tm[:, :, -1].any()))

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
pngs = sorted(glob.glob(OUT + "/*/qc.png"))
cols = 2; rows = (len(pngs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, 3.8 * rows))
for ax in np.ravel(axes): ax.axis("off")
for ax, pth in zip(np.ravel(axes), pngs):
    ax.imshow(mpimg.imread(pth)); ax.set_title(pth.split("/")[-2])
plt.tight_layout(); plt.show()

## 6. Package results for download

In [ ]:
sh(f"cd /kaggle/working && rm -f brats_processed/*/t2_native_ras1mm.nii.gz brats_processed/*/seg_native_ras1mm.nii.gz brats_processed/*/t2_mni_raw.nii.gz brats_processed/*/seg_mni.nii.gz && rm -f brats_processed.zip && zip -qr brats_processed.zip brats_processed && ls -lh brats_processed.zip", check=True)
from IPython.display import FileLink
FileLink("/kaggle/working/brats_processed.zip")